# Text chunking strategies

In [1]:
# Size-delimeted chunking
def chunk_by_char(text: str, max_chunk_size: int = 150, chunk_overlap: int = 20) -> list[str]:
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        
        end = min(start + max_chunk_size, text_length)
        chunk = text[start:end]
        chunks.append(chunk)

        start = (end - chunk_overlap) if end < text_length else text_length

    return chunks

In [2]:
# Chunk by sentence
import re

def chunk_by_sentence(text: str, max_sentences_per_chunk: int = 5, sentence_overlap: int = 1) -> list[str]:
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    start = 0
    num_sentences = len(sentences)

    while start < num_sentences:
        end = min(start + max_sentences_per_chunk, num_sentences)

        chunk = ' '.join(sentences[start:end])
        chunks.append(chunk)

        start += (max_sentences_per_chunk - sentence_overlap)

        if start < 0:
            start = 0

    return chunks


In [3]:
# Chunk by section
def chunk_by_section(text: str, section_delimiter: str = r"\n\n") -> list[str]:
    sections = re.split(section_delimiter, text)
    return sections

In [4]:
with open("./artifacts/report.md", "r") as input_text_file:
    input_text = input_text_file.read()

chunks = chunk_by_char(input_text)

_ = [print(f"Chunk {i}:\n{chunk}\n{'-'*40}\n") for i, chunk in enumerate(chunks)]

Chunk 0:
# **Annual Interdisciplinary Research Review: Cross-Domain Insights**

## Executive Summary

This report synthesizes the key findings and ongoing rese
----------------------------------------

Chunk 1:
ngs and ongoing research efforts across the organization's diverse operational and R&D departments for the past fiscal year. Our strength lies in the 
----------------------------------------

Chunk 2:
trength lies in the cross-pollination of ideas and methodologies, driving innovation and addressing complex challenges that transcend traditional disc
----------------------------------------

Chunk 3:
end traditional disciplinary boundaries. This year's review highlights significant progress in ten critical areas. Advances in **Medical Research** fo
----------------------------------------

Chunk 4:
edical Research** focused on the rare XDR-471 syndrome, yielding new diagnostic insights. Concurrently, **Software Engineering** tackled persistent st
--------------------------------

In [5]:
with open("./artifacts/report.md", "r") as input_text_file:
    input_text = input_text_file.read()

chunks = chunk_by_char(input_text, max_chunk_size=500, chunk_overlap=150)

_ = [print(f"Chunk {i}:\n{chunk}\n{'-'*40}\n") for i, chunk in enumerate(chunks)]

Chunk 0:
# **Annual Interdisciplinary Research Review: Cross-Domain Insights**

## Executive Summary

This report synthesizes the key findings and ongoing research efforts across the organization's diverse operational and R&D departments for the past fiscal year. Our strength lies in the cross-pollination of ideas and methodologies, driving innovation and addressing complex challenges that transcend traditional disciplinary boundaries. This year's review highlights significant progress in ten critical ar
----------------------------------------

Chunk 1:
ddressing complex challenges that transcend traditional disciplinary boundaries. This year's review highlights significant progress in ten critical areas. Advances in **Medical Research** focused on the rare XDR-471 syndrome, yielding new diagnostic insights. Concurrently, **Software Engineering** tackled persistent stability issues, implementing key fixes identified through error code analysis (e.g., `ERR_MEM_ALLOC_FAIL_0x8007000E`). 

In [6]:
with open("./artifacts/report.md", "r") as input_text_file:
    input_text = input_text_file.read()

chunks = chunk_by_sentence(input_text)

_ = [print(f"Chunk {i}:\n{chunk}\n{'-'*40}\n") for i, chunk in enumerate(chunks)]

Chunk 0:
# **Annual Interdisciplinary Research Review: Cross-Domain Insights**

## Executive Summary

This report synthesizes the key findings and ongoing research efforts across the organization's diverse operational and R&D departments for the past fiscal year. Our strength lies in the cross-pollination of ideas and methodologies, driving innovation and addressing complex challenges that transcend traditional disciplinary boundaries. This year's review highlights significant progress in ten critical areas. Advances in **Medical Research** focused on the rare XDR-471 syndrome, yielding new diagnostic insights. Concurrently, **Software Engineering** tackled persistent stability issues, implementing key fixes identified through error code analysis (e.g., `ERR_MEM_ALLOC_FAIL_0x8007000E`).
----------------------------------------

Chunk 1:
Concurrently, **Software Engineering** tackled persistent stability issues, implementing key fixes identified through error code analysis (e.g., `ERR_M

In [7]:
with open("./artifacts/report.md", "r") as input_text_file:
    input_text = input_text_file.read()

chunks = chunk_by_section(input_text, section_delimiter=r"\n## ")

_ = [print(f"Chunk {i}:\n{chunk}\n{'-'*40}\n") for i, chunk in enumerate(chunks)]

Chunk 0:
# **Annual Interdisciplinary Research Review: Cross-Domain Insights**

----------------------------------------

Chunk 1:
Executive Summary

This report synthesizes the key findings and ongoing research efforts across the organization's diverse operational and R&D departments for the past fiscal year. Our strength lies in the cross-pollination of ideas and methodologies, driving innovation and addressing complex challenges that transcend traditional disciplinary boundaries. This year's review highlights significant progress in ten critical areas. Advances in **Medical Research** focused on the rare XDR-471 syndrome, yielding new diagnostic insights. Concurrently, **Software Engineering** tackled persistent stability issues, implementing key fixes identified through error code analysis (e.g., `ERR_MEM_ALLOC_FAIL_0x8007000E`). **Financial Analysis** revealed mixed quarterly performance, prompting strategic reviews, particularly concerning resource allocation impacting R&D pipeli

## Local embeddings with microsoft/harrier-oss-v1-0.6b

First-run downloads ~1.5 GB to `~/.cache/huggingface/hub/`. Subsequent runs load from cache and are near-instant.

In [8]:
# Imports + device auto-detection
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

def pick_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = pick_device()
print(f"Using device: {DEVICE}")

/home/josgood/.cache/pypoetry/virtualenvs/anthropic-course-FdP0x46c-py3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [9]:
# Load the embedding model (first run downloads ~1.5 GB)
MODEL_ID = "microsoft/harrier-oss-v1-0.6b"
print(f"Loading {MODEL_ID} on {DEVICE} (first run downloads ~1.5 GB)...")
model = SentenceTransformer(MODEL_ID, device=DEVICE)
print(f"Embedding dim: {model.get_embedding_dimension()}")

Loading microsoft/harrier-oss-v1-0.6b on cuda (first run downloads ~1.5 GB)...
Embedding dim: 1024


In [10]:
# Asymmetric embedding helpers
# harrier-oss requires prompt_name="web_search_query" on QUERIES but NOT on documents.
def embed_documents(texts: list[str]) -> np.ndarray:
    return model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

def embed_query(query: str) -> np.ndarray:
    return model.encode(
        [query],
        prompt_name="web_search_query",
        normalize_embeddings=True,
    )[0]

In [11]:
# End-to-end retrieval demo: chunk -> embed -> query -> top-k
with open("./artifacts/report.md", "r") as f:
    input_text = f.read()

chunks = chunk_by_section(input_text, section_delimiter=r"\n## ")
doc_vectors = embed_documents(chunks)
print(f"Indexed {len(chunks)} chunks, vectors shape={doc_vectors.shape}")

def retrieve(query: str, k: int = 3) -> list[tuple[float, str]]:
    q = embed_query(query)
    # Both sides L2-normalized -> dot product == cosine similarity
    scores = doc_vectors @ q
    top_idx = np.argsort(-scores)[:k]
    return [(float(scores[i]), chunks[i]) for i in top_idx]

for score, chunk in retrieve("What were the key financial findings?"):
    print(f"[{score:.3f}] {chunk[:200]}...\n{'-'*40}")

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.18s/it]

Indexed 15 chunks, vectors shape=(15, 1024)
[0.492] Section 3: Financial Analysis - Q3 Performance and Outlook

Quarterly financial analysis revealed a complex picture. Overall group revenue saw modest growth of 3.1% year-over-year, primarily driven by...
----------------------------------------
[0.456] Executive Summary

This report synthesizes the key findings and ongoing research efforts across the organization's diverse operational and R&D departments for the past fiscal year. Our strength lies i...
----------------------------------------
[0.420] Section 7: Historical Research - Re-evaluating the Galveston Accords (1921)

Our Historical Research unit undertook a focused analysis of the economic consequences stemming from the Galveston Accords ...
----------------------------------------


In [13]:
print(list(doc_vectors[0]))

[np.float32(-0.01077509), np.float32(-0.036512893), np.float32(-9.453983e-39), np.float32(-0.06283645), np.float32(0.023414213), np.float32(0.11861025), np.float32(0.09127681), np.float32(-0.09029561), np.float32(-0.0015365809), np.float32(-0.03544492), np.float32(0.020243552), np.float32(-0.014191052), np.float32(-0.013509743), np.float32(-4.8113e-40), np.float32(-0.020703288), np.float32(0.012245176), np.float32(-0.2439151), np.float32(0.041667383), np.float32(-0.09546912), np.float32(-0.028313946), np.float32(0.01145604), np.float32(0.0011730721), np.float32(0.0001831629), np.float32(-0.015919583), np.float32(-0.025604093), np.float32(-0.0024010127), np.float32(-0.016817715), np.float32(0.0898757), np.float32(0.026284885), np.float32(0.0053452607), np.float32(0.009254948), np.float32(0.043434836), np.float32(-0.10338742), np.float32(-0.007320788), np.float32(-0.065053724), np.float32(-1.292e-41), np.float32(-0.017212356), np.float32(0.0022680939), np.float32(0.007848961), np.float32

In [12]:
# Optional: feed retrieved context to Claude (closes the RAG loop)
import os
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()

def rag_answer(question: str, k: int = 3) -> str:
    hits = retrieve(question, k=k)
    context = "\n\n---\n\n".join(c for _, c in hits)
    msg = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        messages=[{
            "role": "user",
            "content": f"Use only this context to answer.\n\nContext:\n{context}\n\nQuestion: {question}",
        }],
    )
    return msg.content[0].text

print(rag_answer("What were the key financial findings?"))

## Key Financial Findings

Based on the context provided, the key financial findings from the **Q3 Performance** analysis include:

### Revenue
- **Overall group revenue** grew modestly at **3.1% year-over-year**, driven by strong performance in the primary subsidiary's established markets
- The **emerging markets division contracted by -1.5%**, due to increased competitive pressure and unfavorable currency fluctuations

### Margins & Costs
- **Margin erosion** was observed across several key product lines, linked to:
  - Rising input costs
  - Supply chain disruptions

### Cost Optimization
- **Project Hercules**, aimed at optimizing operational expenditures, generated initial savings, but these were **insufficient to fully offset** the margin pressure

### R&D Investment
- Investment in R&D initiatives remained **stable** but faces a **potential review** given the financial pressures

---

### Outlook & Recommendations
The team recommends a **cautious outlook**, emphasizing:
- Cost c

# Implementing the RAG flow

In [56]:
# VectorIndex implementation
import numpy as np
from typing import Optional, Any, List, Dict, Tuple


class VectorIndex:
    def __init__(
        self,
        distance_metric: str = "cosine",
        embedding_fn=None,
    ):
        self.vectors: List[np.ndarray] = []
        self.documents: List[Dict[str, Any]] = []
        self._vector_dim: Optional[int] = None
        if distance_metric not in ["cosine", "euclidean"]:
            raise ValueError("distance_metric must be 'cosine' or 'euclidean'")
        self._distance_metric = distance_metric
        self._embedding_fn = embedding_fn

    def add_document(self, document: Dict[str, Any]):
        if not self._embedding_fn:
            raise ValueError("Embedding function not provided during initialization.")
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError("Document dictionary must contain a 'content' key.")
        if not isinstance(document["content"], str):
            raise TypeError("Document 'content' must be a string.")

        vector = self._embedding_fn(document["content"])
        self.add_vector(vector=vector, document=document)

    def search(
        self, query: Any, k: int = 1
    ) -> List[Tuple[Dict[str, Any], float]]:
        if not self.vectors:
            return []

        if isinstance(query, str):
            if not self._embedding_fn:
                raise ValueError("Embedding function not provided for string query.")
            query_vector = self._embedding_fn(query)
        elif isinstance(query, (list, np.ndarray)):
            query_vector = np.asarray(query, dtype=float)
        else:
            raise TypeError("Query must be either a string or a list/array of numbers.")

        if self._vector_dim is None:
            return []

        if len(query_vector) != self._vector_dim:
            raise ValueError(
                f"Query vector dimension mismatch. Expected {self._vector_dim}, got {len(query_vector)}"
            )

        if k <= 0:
            raise ValueError("k must be a positive integer.")

        dist_func = self._cosine_distance if self._distance_metric == "cosine" else self._euclidean_distance

        distances = []
        for i, stored_vector in enumerate(self.vectors):
            distance = dist_func(query_vector, stored_vector)
            distances.append((distance, self.documents[i]))

        distances.sort(key=lambda item: item[0])

        return [(doc, dist) for dist, doc in distances[:k]]

    def add_vector(self, vector, document: Dict[str, Any]):
        if not isinstance(vector, (list, np.ndarray)):
            raise TypeError("Vector must be a list or numpy array of numbers.")
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError("Document dictionary must contain a 'content' key.")

        vector = np.asarray(vector, dtype=float)

        if not self.vectors:
            self._vector_dim = len(vector)
        elif len(vector) != self._vector_dim:
            raise ValueError(
                f"Inconsistent vector dimension. Expected {self._vector_dim}, got {len(vector)}"
            )

        self.vectors.append(vector)
        self.documents.append(document)

    def _euclidean_distance(self, vec1: np.ndarray, vec2: np.ndarray) -> float:
        return float(np.linalg.norm(vec1 - vec2))

    def _cosine_distance(self, vec1: np.ndarray, vec2: np.ndarray) -> float:
        norm1 = np.linalg.norm(vec1)
        norm2 = np.linalg.norm(vec2)
        if norm1 == 0 and norm2 == 0:
            return 0.0
        elif norm1 == 0 or norm2 == 0:
            return 1.0
        cosine_similarity = float(np.clip(np.dot(vec1, vec2) / (norm1 * norm2), -1.0, 1.0))
        return 1.0 - cosine_similarity

    def __len__(self) -> int:
        return len(self.vectors)

    def __repr__(self) -> str:
        has_embed_fn = "Yes" if self._embedding_fn else "No"
        return f"VectorIndex(count={len(self)}, dim={self._vector_dim}, metric='{self._distance_metric}', has_embedding_fn='{has_embed_fn}')"


In [57]:
store = VectorIndex(distance_metric="cosine", embedding_fn=embed_query)

for embedding, chunk in zip(doc_vectors, chunks):
    store.add_vector(embedding, {"content": chunk})

In [60]:
results = store.search("What did the software engineering dept do last year?", k=3)

for doc, distance in results:
    print(distance, "\n", doc["content"][0:200])

0.5010982758114375 
 Section 2: Software Engineering - Project Phoenix Stability Enhancements

The Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems
0.5396534685174087 
 Executive Summary

This report synthesizes the key findings and ongoing research efforts across the organization's diverse operational and R&D departments for the past fiscal year. Our strength lies i
0.5542640531997007 
 Section 5: Legal Developments - Navigating IP Precedents and Regulatory Shifts

The Legal department actively monitored and responded to several key developments this year. The ruling in _Synergy Dyna


# BM25 lexical search

In [ ]:
# Author's note:  This performs better than the VoyageAI version, which fails to surface Section 2 in the training video

results = store.search("What happened with INC-2023-Q4-011", k=3)

for doc, distance in results:
    print(distance, "\n", doc["content"][0:200])

0.45430912585541594 
 Section 10: Cybersecurity Analysis - Incident Response Report: INC-2023-Q4-011

The Cybersecurity Operations Center successfully contained and remediated a targeted intrusion attempt tracked as `INC-2
0.5031972293160796 
 Section 2: Software Engineering - Project Phoenix Stability Enhancements

The Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems
0.5800837846090365 
 Section 5: Legal Developments - Navigating IP Precedents and Regulatory Shifts

The Legal department actively monitored and responded to several key developments this year. The ruling in _Synergy Dyna


In [65]:
# BM25Index implementation
import numpy as np
from collections import Counter
from typing import Callable, Optional, Any, List, Dict, Tuple


class BM25Index:
    def __init__(
        self,
        k1: float = 1.5,
        b: float = 0.75,
        tokenizer: Optional[Callable[[str], List[str]]] = None,
    ):
        self.documents: List[Dict[str, Any]] = []
        self._corpus_tokens: List[List[str]] = []
        self._doc_len: List[int] = []
        self._doc_freqs: Dict[str, int] = {}
        self._avg_doc_len: float = 0.0
        self._idf: Dict[str, float] = {}
        self._index_built: bool = False

        self.k1 = k1
        self.b = b
        self._tokenizer = tokenizer if tokenizer else self._default_tokenizer

    def _default_tokenizer(self, text: str) -> List[str]:
        text = text.lower()
        tokens = re.split(r"\W+", text)
        return [token for token in tokens if token]

    def _update_stats_add(self, doc_tokens: List[str]):
        self._doc_len.append(len(doc_tokens))

        seen_in_doc = set()
        for token in doc_tokens:
            if token not in seen_in_doc:
                self._doc_freqs[token] = self._doc_freqs.get(token, 0) + 1
                seen_in_doc.add(token)

        self._index_built = False

    def _calculate_idf(self):
        N = len(self.documents)
        freqs = np.array(list(self._doc_freqs.values()), dtype=float)
        idf_scores = np.log(((N - freqs + 0.5) / (freqs + 0.5)) + 1)
        self._idf = dict(zip(self._doc_freqs.keys(), idf_scores))

    def _build_index(self):
        if not self.documents:
            self._avg_doc_len = 0.0
            self._idf = {}
            self._index_built = True
            return

        self._avg_doc_len = float(np.mean(self._doc_len))
        self._calculate_idf()
        self._index_built = True

    def add_document(self, document: Dict[str, Any]):
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError("Document dictionary must contain a 'content' key.")
        if not isinstance(document["content"], str):
            raise TypeError("Document 'content' must be a string.")

        doc_tokens = self._tokenizer(document["content"])

        self.documents.append(document)
        self._corpus_tokens.append(doc_tokens)
        self._update_stats_add(doc_tokens)

    def _compute_bm25_score(
        self, query_tokens: List[str], doc_index: int
    ) -> float:
        score = 0.0
        doc_term_counts = Counter(self._corpus_tokens[doc_index])
        doc_length = self._doc_len[doc_index]

        for token in query_tokens:
            if token not in self._idf:
                continue

            idf = self._idf[token]
            term_freq = doc_term_counts.get(token, 0)

            numerator = idf * term_freq * (self.k1 + 1)
            denominator = term_freq + self.k1 * (
                1 - self.b + self.b * (doc_length / self._avg_doc_len)
            )
            score += numerator / (denominator + 1e-9)

        return score

    def search(
        self,
        query_text: str,
        k: int = 1,
        score_normalization_factor: float = 0.1,
    ) -> List[Tuple[Dict[str, Any], float]]:
        if not self.documents:
            return []

        if not isinstance(query_text, str):
            raise TypeError("Query text must be a string.")

        if k <= 0:
            raise ValueError("k must be a positive integer.")

        if not self._index_built:
            self._build_index()

        if self._avg_doc_len == 0:
            return []

        query_tokens = self._tokenizer(query_text)
        if not query_tokens:
            return []

        raw_scores = []
        for i in range(len(self.documents)):
            raw_score = self._compute_bm25_score(query_tokens, i)
            if raw_score > 1e-9:
                raw_scores.append((raw_score, self.documents[i]))

        raw_scores.sort(key=lambda item: item[0], reverse=True)

        normalized_results = []
        for raw_score, doc in raw_scores[:k]:
            normalized_score = float(np.exp(-score_normalization_factor * raw_score))
            normalized_results.append((doc, normalized_score))

        normalized_results.sort(key=lambda item: item[1])

        return normalized_results

    def __len__(self) -> int:
        return len(self.documents)

    def __repr__(self) -> str:
        return f"BM25VectorStore(count={len(self)}, k1={self.k1}, b={self.b}, index_built={self._index_built})"


In [64]:
# 1. Chunk your text by sections
#chunks = chunk_by_section(text)

# 2. Create a BM25 store and add documents
store = BM25Index()
for chunk in chunks:
    store.add_document({"content": chunk})

# 3. Search the store
results = store.search("What happened with INC-2023-Q4-011?", 3)

# Print results
for doc, distance in results:
    print(distance, "\n", doc["content"][:200], "\n----\n")

0.2713100078774988 
 Section 2: Software Engineering - Project Phoenix Stability Enhancements

The Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems 
----

0.33172619481009574 
 Section 10: Cybersecurity Analysis - Incident Response Report: INC-2023-Q4-011

The Cybersecurity Operations Center successfully contained and remediated a targeted intrusion attempt tracked as `INC-2 
----

0.9391180106840461 
 Methodology

The insights compiled within this Annual Interdisciplinary Research Review represent a synthesis of findings drawn from standard departmental reporting cycles, specialized project updates 
----



# A Multi-Index RAG pipeline

In [66]:
# Retriever implementation
import numpy as np
from typing import Any, List, Dict, Tuple, Protocol


class SearchIndex(Protocol):
    def add_document(self, document: Dict[str, Any]) -> None: ...

    # Added the 'add_documents' method to avoid rate limiting errors from VoyageAI
    def add_documents(self, documents: List[Dict[str, Any]]) -> None: ...

    def search(
        self, query: Any, k: int = 1
    ) -> List[Tuple[Dict[str, Any], float]]: ...


class Retriever:
    def __init__(self, *indexes: SearchIndex):
        if len(indexes) == 0:
            raise ValueError("At least one index must be provided")
        self._indexes = list(indexes)

    def add_document(self, document: Dict[str, Any]):
        for index in self._indexes:
            index.add_document(document)

    # Added the 'add_documents' method to avoid rate limiting errors from VoyageAI
    def add_documents(self, documents: List[Dict[str, Any]]):
        for index in self._indexes:
            index.add_documents(documents)

    def search(
        self, query_text: str, k: int = 1, k_rrf: int = 60
    ) -> List[Tuple[Dict[str, Any], float]]:
        if not isinstance(query_text, str):
            raise TypeError("Query text must be a string.")
        if k <= 0:
            raise ValueError("k must be a positive integer.")
        if k_rrf < 0:
            raise ValueError("k_rrf must be non-negative.")

        all_results = [
            index.search(query_text, k=k * 5) for index in self._indexes
        ]

        doc_ranks = {}
        for idx, results in enumerate(all_results):
            for rank, (doc, _) in enumerate(results):
                doc_id = id(doc)
                if doc_id not in doc_ranks:
                    doc_ranks[doc_id] = {
                        "doc_obj": doc,
                        "ranks": np.full(len(self._indexes), np.inf),
                    }
                doc_ranks[doc_id]["ranks"][idx] = rank + 1

        def calc_rrf_score(ranks: np.ndarray) -> float:
            finite = ranks[np.isfinite(ranks)]
            return float(np.sum(1.0 / (k_rrf + finite)))

        scored_docs: List[Tuple[Dict[str, Any], float]] = [
            (entry["doc_obj"], calc_rrf_score(entry["ranks"]))
            for entry in doc_ranks.values()
        ]

        filtered_docs = [(doc, score) for doc, score in scored_docs if score > 0]
        if not filtered_docs:
            return []

        docs, scores = zip(*filtered_docs)
        scores_arr = np.array(scores)
        top_idx = np.argsort(-scores_arr)[:k]
        return [(docs[i], float(scores_arr[i])) for i in top_idx]


In [67]:
bm25_index = BM25Index()
vector_index = VectorIndex(distance_metric="cosine", embedding_fn=embed_query)

retriever = Retriever(bm25_index, vector_index)

In [68]:
for chunk in chunks:
    retriever.add_document({"content": chunk})

In [69]:
results = retriever.search("What happened with INC-2023-Q4-011?", k=3)

for doc, score in results:
    print(score, "\n", doc["content"][:200], "\n----\n")

0.03252247488101534 
 Section 2: Software Engineering - Project Phoenix Stability Enhancements

The Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems 
----

0.03252247488101534 
 Section 10: Cybersecurity Analysis - Incident Response Report: INC-2023-Q4-011

The Cybersecurity Operations Center successfully contained and remediated a targeted intrusion attempt tracked as `INC-2 
----

0.03057889822595705 
 Methodology

The insights compiled within this Annual Interdisciplinary Research Review represent a synthesis of findings drawn from standard departmental reporting cycles, specialized project updates 
----

